# Day 34 — Project: a planner agent

Tie Phase 3 together: **plan → execute → trace**, with memory of results. This is your
*Atlas v2* — it decomposes a goal, runs the steps with a tool, and shows a trace you can
inspect. Your task: add tracing around each step. > Needs a provider.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. The solution is below — try first.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator
from shared.tracing import Tracer

tr = Tracer()

def plan(goal):
    raw = chat([{"role": "system", "content": "Return ONLY a JSON array of 2-4 step strings."},
                {"role": "user", "content": goal}], temperature=0).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def do_step(step, context):
    raw = chat([{"role": "system", "content": "Do ONE step. For math reply 'CALC: <expr>', else the result."},
                {"role": "user", "content": f"{context}\nStep: {step}"}], temperature=0).strip()
    return calculator(raw.split(":", 1)[1].strip()) if raw.upper().startswith("CALC:") else raw

def atlas_v2(goal):
    steps, context = plan(goal), ""
    for step in steps:
        # TODO: wrap do_step in a tracer span named after the step (first 20 chars)
        result = do_step(step, context)
        context += f"\n{step} => {result}"
    tr.print_trace()
    return context

# print(atlas_v2("Compute 15% tip on 80, then say the total politely"))

## 🔒 Solution

One correct way to do it.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator
from shared.tracing import Tracer

tr = Tracer()

def plan(goal):
    raw = chat([{"role": "system", "content": "Return ONLY a JSON array of 2-4 step strings."},
                {"role": "user", "content": goal}], temperature=0).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def do_step(step, context):
    raw = chat([{"role": "system", "content": "Do ONE step. For math reply 'CALC: <expr>', else the result."},
                {"role": "user", "content": f"{context}\nStep: {step}"}], temperature=0).strip()
    return calculator(raw.split(":", 1)[1].strip()) if raw.upper().startswith("CALC:") else raw

def atlas_v2(goal):
    steps, context = plan(goal), ""
    for step in steps:
        with tr.span(step[:20]):
            result = do_step(step, context)
        context += f"\n{step} => {result}"
    tr.print_trace()
    return context

print(atlas_v2("Compute a 15% tip on 80, then state the total politely"))